In [1]:
import pandas as pd
from finrl.env.env_stocktrading import StockTradingEnv
from finrl.agents.stablebaselines3.models import DRLAgent
from stable_baselines3 import PPO
import matplotlib.pyplot as plt

import MetaTrader5 as mt5
from datetime import datetime
import pandas as pd
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor
import optuna
import numpy as np
from stable_baselines3.common.evaluation import evaluate_policy
import gymnasium as gym
from gymnasium import spaces
import gc
from wandb.integration.sb3 import WandbCallback
import wandb
from alibi_detect.od import IForest
from stable_baselines3.common.callbacks import BaseCallback
import torch
import os
from stable_baselines3.common.callbacks import EvalCallback
import matplotlib.pyplot as plt
import random
import time

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


ModuleNotFoundError: No module named 'alpaca_trade_api'

In [ ]:
MT5_LOGIN = int(os.environ.get('MT5_LOGIN'))
MT5_PASSWORD = os.environ.get('MT5_PASSWORD')
MT5_SERVER = os.environ.get('MT5_SERVER')
SYMBOL = "EURUSD"
TIMEFRAME = mt5.TIMEFRAME_D1
START = datetime(2020,1,1)
END = datetime(2025,1,1)
MODEL_PATH = "models/best_ppo_forex.zip"
RESUME_TIMESTEPS = 100_000

In [ ]:
# --- Paramètres MT5 ---
SYMBOL = "EURUSD"
TIMEFRAME = mt5.TIMEFRAME_M15
START = pd.Timestamp("2024-01-01")
END = pd.Timestamp("2025-10-01")
PARQUET_PATH = f"data/{SYMBOL.lower()}_{TIMEFRAME}.parquet"


# --- Fonction de connexion ---
def connect_mt5(max_retries=3, wait_time=2):
    """Connexion robuste à MetaTrader 5"""
    print("🔹 Fermeture d'éventuelles connexions précédentes...")
    mt5.shutdown()
    time.sleep(1)

    for attempt in range(1, max_retries + 1):
        print(f"\n🔄 Tentative {attempt}/{max_retries} d'initialisation de MetaTrader 5...")
        if not mt5.initialize():
            err = mt5.last_error()
            print(f"❌ Échec de l'initialisation : {err}")
            time.sleep(wait_time)
            continue

        mt5_version = mt5.version()
        print(f"✅ MetaTrader 5 initialisé : version {mt5_version}")

        if MT5_LOGIN:
            if not mt5.login(login=MT5_LOGIN, password=MT5_PASSWORD, server=MT5_SERVER):
                err = mt5.last_error()
                print(f"❌ Connexion échouée : {err}")
                mt5.shutdown()
                time.sleep(wait_time)
                continue
            print("✅ Connexion réussie à MetaTrader 5.")
        return True

    return False


# --- Chargement ou téléchargement des données ---
if os.path.exists(PARQUET_PATH):
    print(f"📁 Chargement des données existantes depuis {PARQUET_PATH}")
    df_mt5 = pd.read_parquet(PARQUET_PATH)

else:
    print("🌐 Téléchargement des données depuis MetaTrader 5...")
    if connect_mt5():
        rates = mt5.copy_rates_range(SYMBOL, TIMEFRAME, START.to_pydatetime(), END.to_pydatetime())
        if rates is None:
            raise Exception("❌ Aucune donnée reçue depuis MT5.")
        
        df_mt5 = pd.DataFrame(rates)
        if df_mt5.empty:
            raise Exception("❌ Données vides — vérifie la période et le symbole.")

        # Mise en forme
        df_mt5["time"] = pd.to_datetime(df_mt5["time"], unit="s")
        df_mt5.columns = df_mt5.columns.str.lower()  # ✅ Colonnes en minuscules
        df_mt5.reset_index(drop=True, inplace=True)

        print(f"✅ {len(df_mt5)} barres téléchargées.")
        os.makedirs(os.path.dirname(PARQUET_PATH), exist_ok=True)
        df_mt5.to_parquet(PARQUET_PATH, index=False)
        print(f"💾 Données sauvegardées dans {PARQUET_PATH}")
        mt5.shutdown()
    else:
        raise Exception("❌ Impossible de se connecter à MT5.")


# --- Aperçu ---
print(df_mt5.head())